**Mini Project - I**


#module-1


#TrendPulse

**Geetha**

Fetching top story IDs...
Fetching details for 500 stories.

Progress: 500/500 stories processed.

**Success! Total stories saved: 499**

**Fetch Data from API**
Step 1 — Get the list of top story IDs:

returns a JSON array of story IDs (integers). **Fetch the first 500.**

Step 2 — Get each story's details:
Create a folder called data
Save all stories to a file like data/**trends_20240115.json**
Print how many stories were collected in total:

Task 1      →    Task 2      →    Task 3       →    Task 4
Fetch JSON       Clean CSV        NumPy/Pandas      Visualise


This project follows a pipeline approach:
# Step 1: Collect data from API
# Step 2: Clean the data
# Step 3: Analyze trends
# Step 4: Visualize insights

** Get the list of top story IDs:**

https://hacker-news.firebaseio.com/v0/topstories.json
This returns a JSON array of story IDs (integers). Fetch the first 500.

In [ ]:
#%%writefile fetch_hacker_news.py
import requests
import json
import os
import time

from datetime import datetime
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- Constants & Categories ---
DATA_FOLDER = "data"
HEADERS = {"User-Agent": "TrendPulse/1.0"}
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

def fetch_hacker_news():
    if not os.path.exists(DATA_FOLDER):
        os.makedirs(DATA_FOLDER)

    # 1. LOGIC: Create a robust session with retries
    session = requests.Session()
    retry_strategy = Retry(
        total=5,  # Increased retries for stability
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504]
    )
    session.mount("https://", HTTPAdapter(max_retries=retry_strategy))

    try:
        print("Fetching top story IDs...")
        id_response = session.get(TOP_STORIES_URL, headers=HEADERS, timeout=10)
        id_response.raise_for_status()
        story_ids = id_response.json()[:500]

        all_stories = []

        print(f"Fetching details for {len(story_ids)} stories. This will take ~1 minute...")

        for count, s_id in enumerate(story_ids, 1):
            try:
                # 2. LOGIC: Fetch each story using the session
                item_response = session.get(ITEM_URL.format(s_id), headers=HEADERS, timeout=10)
                story_data = item_response.json()
                # Inside your Task 1 loop

                if story_data and story_data.get('type') == 'story':
                    title = story_data.get('title', '')
                    all_stories.append({
                        "post_id": s_id,
                        "title": title,
                        "category": "uncategorized", # Placeholder for your categorization logic
                        "score": story_data.get('score', 0),
                        #"url": story_data.get('url', 'N/A'),
                        "collected_at": datetime.now().isoformat()
                    })

                # 3. LOGIC: Small sleep to prevent 'UNEXPECTED_EOF'
                time.sleep(0.2)

            except Exception as e:
                print(f"\nSkipping ID {s_id} due to connection blip: {e}")
                continue

            if count % 50 == 0:
                print(f"Progress: {count}/500 stories processed.")

        # 4. SAVE DATA
        date_str = datetime.now().strftime("%Y%m%d")
        filename = f"{DATA_FOLDER}/trends_{date_str}.json"
        with open(filename, 'w') as f:
            json.dump(all_stories, f, indent=4)

        print(f"\nSuccess! Total stories saved: {len(all_stories)}")

    except Exception as e:
        print(f"Critical Pipeline Error: {e}")

if __name__ == "__main__":
    fetch_hacker_news()

Fetching top story IDs...
Fetching details for 500 stories. This will take ~1 minute...
Progress: 50/500 stories processed.
Progress: 100/500 stories processed.
Progress: 150/500 stories processed.
Progress: 200/500 stories processed.
Progress: 250/500 stories processed.
Progress: 300/500 stories processed.
Progress: 350/500 stories processed.
Progress: 400/500 stories processed.
Progress: 450/500 stories processed.
Progress: 500/500 stories processed.

Success! Total stories saved: 499


First, generate the JSON
python task1_data_collection.py

 **Second, process that JSON into a CSV**
python task2_data_processing.py

Take first 500 IDs
3. Loop through each ID
4. Fetch story details
5. Extract title + info
6. Assign category using keywords
7. Store results
8. Save to CSV

In [ ]:
#task 2 2a,b,c)
#dataprocessing
#%%writefile task2_data_processing.py
import requests
import json
import os
import time
from datetime import datetime

class TrendPulseCollector:
    def __init__(self):
        self.base_url = "https://hacker-news.firebaseio.com/v0"
        self.output_dir = "data"
        self.categories = {
            "technology": ["software", "ai", "apple", "google", "programming", "web"],
            "science": ["space", "nasa", "physics", "biology", "research", "climate"],
            "entertainment": ["movie", "music", "game", "nintendo", "film", "art"],
            "worldnews": ["china", "europe", "policy", "court", "government", "war"],
            "sports": ["f1", "chess", "football", "soccer", "olympics", "nba"]
        }

        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)

    def fetch_story_details(self, story_id, category):
        """Extracts the 7 required fields from a story ID."""
        try:
            url = f"{self.base_url}/item/{story_id}.json"
            response = requests.get(url)
            response.raise_for_status()
            item = response.json()

            if item and item.get('type') == 'story' and 'title' in item:
                # Required Fields (Task 2 Mapping)
                return {
                    "post_id": item.get("id"),
                    "title": item.get("title"),
                    "category": category,
                    "score": item.get("score", 0),
                    "num_comments": item.get("descendants", 0),
                    "author": item.get("by"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }
        except Exception as e:
            print(f"  [!] Failed to fetch story {story_id}: {e}")
        return None

    def collect_all(self):
        print("Starting Task 1: Collection...")
        all_collected_stories = []


        # STEP 1: Increase the pool by fetching from 3 different endpoints
        # This gives us ~1,500 IDs to search through instead of just 500
        endpoints = ["topstories", "newstories", "beststories"]
        combined_ids = []

        for endpoint in endpoints:
            try:
                url = f"{self.base_url}/{endpoint}.json"
                ids = requests.get(url, timeout=10).json()
                combined_ids.extend(ids)
            except Exception as e:
                print(f"  [!] Could not fetch {endpoint}: {e}")

        # Remove duplicate IDs to save time
        unique_ids = list(set(combined_ids))
        print(f"[*] Total unique IDs to scan: {len(unique_ids)}")
        self.categories = {
            "technology": ["software", "ai", "web", "programming", "tech", "app", "data"],
            "science": ["science", "space", "nature", "research", "study", "physics", "health"],
            "entertainment": ["game", "music", "movie", "art", "video", "culture", "media", "book"],
            "worldnews": ["news", "world", "policy", "law", "china", "global", "report", "court"],
            "sports": ["sport", "ball", "chess", "race", "f1", "match", "club", "win", "team"]
        }

        for category, keywords in self.categories.items():
            print(f"Searching for {category}...")
            category_count = 0

            for story_id in unique_ids:
                if category_count >= 25: # Limit to 25 per category
                    break

                # Peek at the story to see if it matches keywords
                story_url = f"{self.base_url}/item/{story_id}.json"
                item = requests.get(story_url).json()

                if item and 'title' in item:
                    title_lower = item['title'].lower()
                    # Check if any keyword for this category is in the title
                    if any(kw in title_lower for kw in keywords):
                        story_data = self.fetch_story_details(story_id, category)
                        if story_data:
                            all_collected_stories.append(story_data)
                            category_count += 1

            # REQUIREMENT: Wait 2 seconds between each category
            time.sleep(2)

        return all_collected_stories

    def save_to_json(self, data):
        timestamp = datetime.now().strftime("%Y%m%d")
        filename = f"data/trends_{timestamp}.json"

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)

        print(f"\nTask 1 Complete!")
        print(f"Collected {len(data)} stories. Saved to {filename}")

if __name__ == "__main__":
    collector = TrendPulseCollector()
    stories = collector.collect_all()
    collector.save_to_json(stories)



Starting Task 1: Collection...
[*] Total unique IDs to scan: 976
Searching for technology...
Searching for science...
Searching for entertainment...
Searching for worldnews...
Searching for sports...

Task 1 Complete!
Collected 123 stories. Saved to data/trends_20260409.json


A file at data/trends_YYYYMMDD.json
At least 100 stories inside it
A console message like: Collected 122 stories. Saved to data/trends_20240115.json


In [ ]:
#Task1


In [ ]:
import pandas as pd
import os

def run_analysis():
    # --- 1. Load the Cleaned Data ---
    input_file = "data/trends_clean.csv"

    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found. Please run Task 2 first.")
        return

    # Load the CSV into a DataFrame
    df = pd.read_csv(input_file)
    print(f"Successfully loaded {len(df)} stories for analysis.\n")

    # --- 2. Perform Data Analysis ---

    # Analysis A: Average Score per Category
    # We group by 'category' and calculate the mean of the 'score' column
    avg_scores = df.groupby('category')['score'].mean().sort_values(ascending=False)

    # Analysis B: Total Comments per Category
    # We sum the 'num_comments' for each category
    total_comments = df.groupby('category')['num_comments'].sum().sort_values(ascending=False)

    # Analysis C: Top 5 Highest Scoring Stories
    # We sort by 'score' and take the first 5 rows
    top_5_stories = df.nlargest(5, 'score')[['title', 'score', 'category']]

    # --- 3. Display Results to Console ---
    print("--- Average Score per Category ---")
    print(avg_scores)
    print("\n--- Total Comments per Category ---")
    print(total_comments)
    print("\n--- Top 5 Stories by Score ---")
    print(top_5_stories)

    # --- 4. Save Results to a Text File (Required for Submission) ---
    output_file = "data/analysis_summary.txt"
    with open(output_file, "w") as f:
        f.write("TrendPulse Analysis Summary\n")
        f.write("===========================\n\n")

        f.write("1. Average Score per Category:\n")
        f.write(avg_scores.to_string() + "\n\n")

        f.write("2. Total Comments per Category:\n")
        f.write(total_comments.to_string() + "\n\n")

        f.write("3. Top 5 Stories by Score:\n")
        f.write(top_5_stories.to_string(index=False) + "\n")

    print(f"\n Analysis complete. Summary saved to {output_file}")

if __name__ == "__main__":
    run_analysis()

Error: data/trends_clean.csv not found. Please run Task 2 first.


In [ ]:
# Fetch Data from API
#make api call
import requests
import json
import os
import time
from datetime import datetime

# Logic: Define constants for the project requirements
DATA_FOLDER = "data"
HEADERS = {"User-Agent": "TrendPulse/1.0"}
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

# Logic: Mapping keywords for categorization (Case-Insensitive)
CATEGORIES = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "accesory","gpu", "llm","hardware","cpu","ram","rom","api","ipad","calc","laptop","chip"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "new born","global","poll","season","ntv","sensor","claude","flow""helio","helicoptor"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship","indoor","cricket","carom","chess","football","gunshoot","golf","race","bike","himalayan"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome","rocket","genetics","blood","iv","lazer""zoology","plant","cosmetics","rbc","platelets"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming","tv","amazon","hotstar","prime","ludo","buffer","sun","cards","swimming","run"]
}

def get_category(title):
    """Logic: Returns the first matching category based on keywords in title."""
    if not title: return None
    title_lower = title.lower()
    for category, keywords in CATEGORIES.items():
        if any(kw in title_lower for kw in keywords):
            return category
    return None

def fetch_trendpulse_data():
    # 1. SETUP: Create folder if it doesn't exist (5 marks requirement)
    if not os.path.exists(DATA_FOLDER):
        os.makedirs(DATA_FOLDER)

    all_stories = []

    try:
        # 2. STEP 1: Get Top 500 IDs
        print("Connecting to HackerNews...")
        id_response = requests.get(TOP_STORIES_URL, headers=HEADERS, timeout=5)
        id_response.raise_for_status()
        story_ids = id_response.json()[:500]

        # 3. STEP 2: Fetch and Categorize (Logic: Max 25 per category)
        for cat_name, keywords in CATEGORIES.items():
            print(f"Fetching stories for category: {cat_name}...")
            count_in_category = 0

            for s_id in story_ids:
                if count_in_category >= 25:
                    break # Stop when we hit the limit for this category

                try:
                    # Request individual story details
                    item_res = requests.get(ITEM_URL.format(s_id), headers=HEADERS, timeout=5)
                    item_data = item_res.json()

                    # Logic: Validate type and check if it belongs in current category
                    if item_data and item_data.get('type') == 'story':
                        title = item_data.get('title', '')
                        if get_category(title) == cat_name:
                            # 4. FIELD EXTRACTION (7 marks requirement)
                            story_entry = {
                                "post_id": item_data.get("id"),
                                "title": title,
                                "category": cat_name,
                                "score": item_data.get("score", 0),
                                "num_comments": item_data.get("descendants", 0),
                                "author": item_data.get("by", "unknown"),
                                "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                            }
                            all_stories.append(story_entry)
                            count_in_category += 1

                except Exception as e:
                    print(f"Skipping ID {s_id} due to error: {e}")
                    continue

            # 5. POLITE FETCHING (8 marks requirement: 2s sleep per category)
            print(f"Finished {cat_name}. Found {count_in_category} stories. Pausing...")
            time.sleep(2)

        # 6. SAVE TO JSON (5 marks requirement)
        # Naming: trends_YYYYMMDD.json
        date_str = datetime.now().strftime("%Y%m%d")
        filename = f"{DATA_FOLDER}/trends_{date_str}.json"

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(all_stories, f, indent=4)

        # 7. FINAL CONSOLE MESSAGE (Required Output)
        print(f"Collected {len(all_stories)} stories. Saved to {filename}")

    except Exception as e:
        print(f"Critical failure in data collection: {e}")

if __name__ == "__main__":
    fetch_trendpulse_data()

Connecting to HackerNews...
Fetching stories for category: technology...
Finished technology. Found 25 stories. Pausing...
Fetching stories for category: worldnews...
Finished worldnews. Found 18 stories. Pausing...
Fetching stories for category: sports...
Finished sports. Found 6 stories. Pausing...
Fetching stories for category: science...
Finished science. Found 25 stories. Pausing...
Fetching stories for category: entertainment...
Finished entertainment. Found 25 stories. Pausing...
Collected 99 stories. Saved to data/trends_20260404.json


 Script runs without errors
 JSON file saved in data/ folder
 At least 100 stories collected
 Each story has all 7 required fields
 Code is commented.

 #Collected 99 stories. Saved to data/trends_20260404.json

In [10]:
#loaded raw data
import pandas as pd
import glob
import os

# Logic: Define the directory where Task 1 saved the data
DATA_DIR = "data"

def load_trending_data():
    """
    Logic: This function finds the latest JSON file and loads it into Pandas.
    Requirement: 4 marks for loading and printing the row count.
    """

    # 1. FIND THE FILE
    # Logic: Search for any JSON file in the data folder.
    json_files = glob.glob(os.path.join(DATA_DIR, "*.json"))

    if not json_files:
        print("Error: No JSON files found. Did you run Task 1 first?")
        return None

    # Logic: Pick the most recent file if multiple exist
    latest_file = max(json_files, key=os.path.getctime)
    print(f"Loading data from: {latest_file}")

    # 2. LOAD INTO PANDAS
    try:
        # Requirement: Load the JSON file into a DataFrame
        df = pd.read_json(latest_file)

        # 3. PRINT ROW COUNT
        # Requirement: Print how many rows were loaded
        print(f"Successfully loaded {len(df)} rows from the JSON file.")

        return df

    except Exception as e:
        print(f"Failed to load JSON: {e}")
        return None

if __name__ == "__main__":
    df_raw = load_trending_data()

Loading data from: data/trends_20260409.json
Successfully loaded 107 rows from the JSON file.


In [ ]:
#expected output
#final
import pandas as pd
import glob
import os

def process_data():
    # 1. FIND AND LOAD
    data_dir = "data"
    json_files = glob.glob(os.path.join(data_dir, "*.json"))
    if not json_files:
        print("No JSON files found!")
        return

    latest_file = max(json_files, key=os.path.getctime)
    df = pd.read_json(latest_file)

    # REQUIRED OUTPUT: Print initial load
    print(f"Loaded {len(df)} stories from {latest_file}")

    # 2. STEP-BY-STEP CLEANING (Logic: Sequential filtering)

    # Remove Duplicates
    df = df.drop_duplicates(subset=['post_id'])
    print(f"After removing duplicates: {len(df)}")

    # Remove Nulls
    df = df.dropna(subset=['post_id', 'title', 'score'])
    print(f"After removing nulls: {len(df)}")

    # Remove Low Scores
    df = df[df['score'] >= 5]
    print(f"After removing low scores: {len(df)}")

    # 3. SAVE AND SUMMARIZE
    output_path = os.path.join(data_dir, "trends_clean.csv")
    df.to_csv(output_path, index=False)

    # REQUIRED OUTPUT: Final save confirmation
    print(f"Saved {len(df)} rows to {output_path}")

    # REQUIRED OUTPUT: Stories per category
    print("\nStories per category:")
    print(df['category'].value_counts())

if __name__ == "__main__":
    process_data()
if os.path.exists("data/trends_clean.csv"):
    print("\nVerification Success: trends_clean.csv exists in the data folder.")
else:
    print("\n Verification Failed: File not found.")
    #Verification Success: trends_clean.csv exists in the data folder.

Loaded 125 stories from data/trends_20260404.json
After removing duplicates: 113
After removing nulls: 113
After removing low scores: 66
Saved 66 rows to data/trends_clean.csv

Stories per category:
category
technology       17
entertainment    14
worldnews        14
science          11
sports           10
Name: count, dtype: int64

Verification Success: trends_clean.csv exists in the data folder.


In [ ]:
from google.colab import files
files.download("data/trends_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Loaded 125 stories from data/trends_20260404.json
After removing duplicates: 113
After removing nulls: 113
After removing low scores: 66
**Saved** 66 rows to data/trends_clean.csv

Stories per category:
category
technology       17
entertainment    14
worldnews        14
science          11
sports           10
**Name: count, dtype: int64**

In [ ]:

import json

# Construct the filename using the current date, as in the previous execution
from datetime import datetime
date_str = datetime.now().strftime("%Y%m%d")
filename = f"data/trends_{date_str}.json"

# Load the JSON file
with open(filename, 'r', encoding='utf-8') as f:
    loaded_stories = json.load(f)

# Display the first few stories to verify
print(f"Loaded {len(loaded_stories)} stories from {filename}")
if loaded_stories:
    print("First 3 loaded stories:")
    for i, story in enumerate(loaded_stories[:3]):
        print(json.dumps(story, indent=2))
else:
    print("No stories loaded or the file is empty.")

Loaded 499 stories from data/trends_20260404.json
First 3 loaded stories:
{
  "post_id": 47640728,
  "title": "Show HN: A game where you build a GPU",
  "category": "uncategorized",
  "score": 122,
  "collected_at": "2026-04-04T18:13:51.853153"
}
{
  "post_id": 47637757,
  "title": "Simple self-distillation improves code generation",
  "category": "uncategorized",
  "score": 374,
  "collected_at": "2026-04-04T18:13:52.061121"
}
{
  "post_id": 47639567,
  "title": "Show HN: TurboQuant-WASM \u2013 Google's vector quantization in the browser",
  "category": "uncategorized",
  "score": 59,
  "collected_at": "2026-04-04T18:13:52.266630"
}


import the trends_20260404.json file by loading its content into a Python variable and then display its first few entries.

In [ ]:
from google.colab import files
files.download("data/trends_20260404.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Create a folder called data/ if it doesn't exist
Save all stories to a file like data/trends_20240115.json
Print how many stories were collected in total: 99 stories collected and saved.

In [ ]:
#make api call
import requests
import json
import os
import time
from datetime import datetime

# Logic: Define constants for the project requirements
DATA_FOLDER = "data"
HEADERS = {"User-Agent": "TrendPulse/1.0"}
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

# Logic: Mapping keywords for categorization (Case-Insensitive)
CATEGORIES = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "accesory","gpu", "llm","hardware","cpu","ram","rom","api","ipad","calc","laptop","chip"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "new born","global","poll","season","ntv","sensor","claude","flow""helio","helicoptor"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship","indoor","cricket","carom","chess","football","gunshoot","golf","race","bike","himalayan"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome","rocket","genetics","blood","iv","lazer""zoology","plant","cosmetics","rbc","platelets"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming","tv","amazon","hotstar","prime","ludo","buffer","sun","cards","swimming","run"]
}

def get_category(title):
    """Logic: Returns the first matching category based on keywords in title."""
    if not title: return None
    title_lower = title.lower()
    for category, keywords in CATEGORIES.items():
        if any(kw in title_lower for kw in keywords):
            return category
    return None

def fetch_trendpulse_data():
    # 1. SETUP: Create folder if it doesn't exist (5 marks requirement)
    if not os.path.exists(DATA_FOLDER):
        os.makedirs(DATA_FOLDER)

    all_stories = []

    try:
        # 2. STEP 1: Get Top 500 IDs
        print("Connecting to HackerNews...")
        id_response = requests.get(TOP_STORIES_URL, headers=HEADERS, timeout=2)
        id_response.raise_for_status()
        story_ids = id_response.json()[:500]

        # 3. STEP 2: Fetch and Categorize (Logic: Max 25 p r category)
        for cat_name, keywords in CATEGORIES.items():
            print(f"Fetching stories for category: {cat_name}...")
            count_in_category = 0

            for s_id in story_ids:
                if count_in_category >= 25:
                    break # Stop when we hit the limit for this category

                try:
                    # Request individual story details
                    item_res = requests.get(ITEM_URL.format(s_id), headers=HEADERS, timeout=2)
                    item_data = item_res.json()

                    # Logic: Validate type and check if it belongs in current category
                    if item_data and item_data.get('type') == 'story':
                        title = item_data.get('title', '')
                        if get_category(title) == cat_name:
                            # 4. FIELD EXTRACTION (7 marks requirement)
                            story_entry = {
                                "post_id": item_data.get("id"),
                                "title": title,
                                "category": cat_name,
                                "score": item_data.get("score", 0),
                                "num_comments": item_data.get("descendants", 0),
                                "author": item_data.get("by", "unknown"),
                                "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                            }
                            all_stories.append(story_entry)
                            count_in_category += 1

                except Exception as e:
                    print(f"Skipping ID {s_id} due to error: {e}")
                    continue

            # 5. POLITE FETCHING (8 marks requirement: 2s sleep per category)
            print(f"Finished {cat_name}. Found {count_in_category} stories. Pausing...")
            time.sleep(2)

        # 6. SAVE TO JSON (5 marks requirement)
        # Naming: trends_YYYYMMDD.json
        date_str = datetime.now().strftime("%Y%m%d")
        filename = f"{DATA_FOLDER}/trends_{date_str}.json"

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(all_stories, f, indent=4)

        # 7. FINAL CONSOLE MESSAGE (Required Output)
        print(f"Collected {len(all_stories)} stories. Saved to {filename}")

    except Exception as e:
        print(f"Critical failure in data collection: {e}")

if __name__ == "__main__":
    fetch_trendpulse_data()

Connecting to HackerNews...
top 5 stories
Fetching stories for category: technology...
Finished technology. Found 25 stories. Pausing...
Fetching stories for category: worldnews...
Skipping ID 47703707 due to error: HTTPSConnectionPool(host='hacker-news.firebaseio.com', port=443): Read timed out. (read timeout=2)
Finished worldnews. Found 21 stories. Pausing...
Fetching stories for category: sports...
Finished sports. Found 11 stories. Pausing...
Fetching stories for category: science...
Skipping ID 47696210 due to error: HTTPSConnectionPool(host='hacker-news.firebaseio.com', port=443): Read timed out. (read timeout=2)
Finished science. Found 25 stories. Pausing...
Fetching stories for category: entertainment...
Finished entertainment. Found 25 stories. Pausing...
Collected 107 stories. Saved to data/trends_20260409.json


In [ ]:
import requests

# Define the endpoint
url = "https://hacker-news.firebaseio.com/v0/topstories.json"

# Define the required header
headers = {"User-Agent": "TrendPulse/1.0"}

try:
    # Make the GET request
    response = requests.get(url, headers=headers)

    # Check if the request was successful
    response.raise_for_status()

    # Convert the response to a list and slice the first 500
    top_500_ids = response.json()[:500]

    print(f"Successfully retrieved {len(top_500_ids)} story IDs.")
    print(f"First 5 IDs: {top_500_ids[:5]}")

except Exception as e:
    print(f"An error occurred: {e}")

Successfully retrieved 500 story IDs.
First 5 IDs: [47706268, 47701148, 47706611, 47685644, 47661977]


In [ ]:
import requests
import json

# Configuration
HEADERS = {"User-Agent": "TrendPulse/1.0"}
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL_TEMPLATE = "https://hacker-news.firebaseio.com/v0/item/{}.json"

def print_top_stories(limit=5):
    try:
        # 1. Fetch the list of Top IDs
        id_response = requests.get(TOP_STORIES_URL, headers=HEADERS)
        id_response.raise_for_status()

        # Slice to get only the top 5
        top_ids = id_response.json()[:limit]

        print(f"--- Top {limit} Stories on Hacker News ---\n")

        # 2. Iterate through the IDs to get full details
        for story_id in top_ids:
            try:
                # Fetch full story object
                item_url = ITEM_URL_TEMPLATE.format(story_id)
                item_response = requests.get(item_url, headers=HEADERS)
                item_response.raise_for_status()

                story_data = item_response.json()

                # 3. Print the ID and the full JSON object
                # Using json.dumps for a clean, readable print (pretty-print)
                print(f"ID: {story_id}")
                print(json.dumps(story_data, indent=4))
                print("-" * 40)

            except Exception as e:
                print(f"Could not fetch details for ID {story_id}: {e}")

    except Exception as e:
        print(f"Failed to fetch top stories list: {e}")

if __name__ == "__main__":
    print_top_stories()

--- Top 5 Stories on Hacker News ---

ID: 47706268
{
    "by": "gregsadetsky",
    "descendants": 161,
    "id": 47706268,
    "kids": [
        47707031,
        47707087,
        47706557,
        47706531,
        47707130,
        47706564,
        47707185,
        47706225,
        47706717,
        47706854,
        47707038,
        47707128,
        47706941,
        47706766,
        47706784,
        47707083,
        47706845,
        47706522,
        47707089,
        47706858,
        47706519,
        47706777,
        47706492,
        47706497,
        47706771,
        47707014,
        47707124,
        47706513,
        47706455,
        47706762,
        47707102,
        47706217,
        47706456,
        47706801,
        47706351,
        47706562,
        47706355,
        47706721,
        47707080,
        47706445,
        47706714,
        47706759,
        47706853,
        47706779,
        47706950,
        47706609,
        47706676
    ],
    "score"

When you run this, Hacker News will return a dictionary for each story. The most common fields include:

**id:** The item's unique ID (which you specifically requested).

**title**: The title of the story.

**url:** The URL of the story.

score: **bold text** The story's score (upvotes).

**by:** The username of the item's author.

descendants **bold text**: In the HN API, this represents the total comment count.

**time**: Creation date of the item in Unix Time (seconds).

**type:** The type of item (usually "story").

**output**
Connecting to HackerNews...
Fetching stories for category: technology...
`Finished technology. Found 25 stories`. Pausing...
Fetching stories for category: worldnews...
`Finished worldnews. Found 19 stories.` Pausing...
Fetching stories for category: sports...
`Finished sports. Found 7 stories`. Pausing...
Fetching stories for category: science...
`Finished science. Found 25 stories.` Pausing...
Fetching stories for category: entertainment...
`Finished entertainment. Found 25 stories`. Pausing...
#Collected 101 stories. Saved to data/trends_20260404.json

In [ ]:
#analysis
#%%writefile run_analysis.py
import pandas as pd
import numpy as np
import os

def run_analysis():
    # --- 1. Load and Explore (4 marks) ---
    input_file = "data/trends_clean.csv"

    # Check if file exists before loading
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found. Please run Task 2 first.")
        return

    # Load into DataFrame
    df = pd.read_csv(input_file)
    print(f"\nShape of DataFrame: {df.shape}")
    print("\n--- First 5 Rows of Cleaned Data ---")

    print(df.head())



    # Calculate averages using Pandas
    avg_score_val = df['score'].mean()
    avg_comments_val = df['num_comments'].mean()
    print(f"Average Score: {avg_score_val:.2f}")
    print(f"Average Comments: {avg_comments_val:.2f}\n")

    # --- 2. Basic Analysis with NumPy (8 marks) ---
    # Converting columns to NumPy arrays for specific math operations
    scores_array = df['score'].values
    comments_array = df['num_comments'].values

    print("--- NumPy Statistical Analysis ---")
    print(f"Mean Score: {np.mean(scores_array):.2f}")
    print(f"Median Score: {np.median(scores_array)}")
    print(f"Standard Deviation: {np.std(scores_array):.2f}")
    print(f"Highest Score: {np.max(scores_array)}")
    print(f"Lowest Score: {np.min(scores_array)}")

    # Category with most stories (using Pandas mode which utilizes NumPy logic)
    most_common_category = df['category'].mode()[0]
    print(f"Category with most stories: {most_common_category}")

    # Identify story with most comments using NumPy argmax
    max_comments_idx = np.argmax(comments_array)
    top_story_title = df.iloc[max_comments_idx]['title']
    top_story_count = df.iloc[max_comments_idx]['num_comments']
    print(f"Story with most comments: '{top_story_title}' ({top_story_count} comments)\n")

    # --- 3. Add New Columns (5 marks) ---
    # Formula: engagement = num_comments / (score + 1)
    # Adding 1 to denominator prevents "Division by Zero" errors
    df['engagement'] = df['num_comments'] / (df['score'] + 1)

    # Formula: is_popular = True if score > average score
    df['is_popular'] = df['score'] > avg_score_val

    # --- 4. Save the Result (3 marks) ---
    output_file = "data/trends_analysed.csv"

    # Ensure the data directory exists just in case
    if not os.path.exists('data'):
        os.makedirs('data')

    df.to_csv(output_file, index=False)

    print("--- Task 3 Complete ---")
    print(f" Successfully saved updated analysis to {output_file}")
    print(f"New columns 'engagement' and 'is_popular' added to {len(df)} rows.")

if __name__ == "__main__":
    run_analysis()


Shape of DataFrame: (66, 7)

--- First 5 Rows of Cleaned Data ---
    post_id                                              title    category  \
0  47611921                       Significant raise of reports  technology   
1  47603737       AI for American-produced cement and concrete  technology   
2  47620155  Marc Andreessen Is Right That AI Isn't Killing...  technology   
3  47618123  Priced Out by AI: The Memory Chip Crisis Hitti...  technology   
4  47636556                      The Web Is an Antitrust Wedge  technology   

   score  num_comments          author         collected_at  
0    310           155      stratos123  2026-04-04 19:04:50  
1    223           117        latchkey  2026-04-04 19:04:50  
2     38             9    bigbobbeeper  2026-04-04 19:04:50  
3      6             0             dxs  2026-04-04 19:04:50  
4      6             0  genericlemon24  2026-04-04 19:04:51  
Average Score: 98.58
Average Comments: 57.15

--- NumPy Statistical Analysis ---
Mean Score:

In [14]:
files.download("data/trends_analysed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Shape of DataFrame: (66, 7)

--- First 5 Rows of Cleaned Data ---
    post_id                                              title    category  \
0  47611921                       Significant raise of reports  technology   
1  47603737       AI for American-produced cement and concrete  technology   
2  47620155  Marc Andreessen Is Right That AI Isn't Killing...  technology   
3  47618123  Priced Out by AI: The Memory Chip Crisis Hitti...  technology   
4  47636556                      The Web Is an Antitrust Wedge  technology   

   score  num_comments          author         collected_at  
0    310           155      stratos123  2026-04-04 19:04:50  
1    223           117        latchkey  2026-04-04 19:04:50  
2     38             9    bigbobbeeper  2026-04-04 19:04:50  
3      6             0             dxs  2026-04-04 19:04:50  
4      6             0  genericlemon24  2026-04-04 19:04:51  
Average Score: 98.58
Average Comments: 57.15

--- NumPy Statistical Analysis ---
Mean Score: 98.58
Median Score: 28.0
Standard Deviation: 163.54
Highest Score: 957
Lowest Score: 5
Category with most stories: technology
Story with most comments: 'Tell HN: Anthropic no longer allowing Claude Code subscriptions to use OpenClaw' (735 comments)

--- Task 3 Complete ---
 Successfully saved updated analysis to data/trends_analysed.csv
New columns 'engagement' and 'is_popular' added to 66 rows.

In [8]:
#task3
import pandas as pd
import os

# --- CONFIGURATION ---
INPUT_FILE = "data/trends_clean.csv"
OUTPUT_FILE = "data/analysis_results.csv"

def run_analysis():
    """
    Logic: Performs statistical analysis on cleaned trending data.
    Requirement: 20 Marks for Top 5, Category Averages, and Engagement Ratios.
    """

    # 1. LOAD DATA
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found. Please run Task 2 first.")
        return

    df = pd.read_csv(INPUT_FILE)
    print(f"--- Analysis started on {len(df)} stories ---\n")

    # 2. TOP 5 STORIES BY SCORE (7 Marks Logic)
    # Requirement: Find the top 5 most upvoted stories
    print("Top 5 Stories by Score:")
    top_5 = df.nlargest(5, 'score')[['title', 'score', 'category']]
    print(top_5)
    print("*" * 30)

    # 3. CATEGORY AVERAGES (7 Marks Logic)
    # Requirement: Calculate the average score for every category
    # Logic: Use groupby to see which category is performing best overall
    print("\nAverage Score per Category:")
    avg_scores = df.groupby('category')['score'].mean().sort_values(ascending=False).round(2)
    print(avg_scores)
    print("*" * 30)

    # 4. ENGAGEMENT RATIO (6 Marks Logic)
    # Requirement: Create a ratio of comments to upvotes
    # Logic: Avoid division by zero by ensuring score is at least 1 (already handled in Task 2)
    df['engagement_ratio'] = (df['num_comments'] / df['score']).round(2)

    print("\nTop 3 Stories by Engagement Ratio (Comments/Upvotes):")
    top_engagement = df.nlargest(3, 'engagement_ratio')[['title', 'engagement_ratio']]
    print(top_engagement)

    # 5. SAVE RESULTS
    # Logic: Save the new dataframe (with the ratio column) for Task 4 visualization
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nTask 3 Completed. Results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    run_analysis()

Error: data/trends_clean.csv not found. Please run Task 2 first.


In [ ]:
from google.colab import files
files.download("data/analysis_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#%%writefile task4_visualization.py
#task4-visualization
import pandas as pd
import matplotlib.pyplot as plt
import os

def run_visualization():
    # --- Task 1: Setup (2 marks) ---
    input_file = "data/trends_analysed.csv"

    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found. Please run Task 3 first.")
        return

    # Load the analysed data
    df = pd.read_csv(input_file)

    # Create output directory
    output_dir = "outputs"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created {output_dir}/ folder.")

    # --- Task 2: Chart 1 - Top 10 Stories by Score (6 marks) ---
    plt.figure(figsize=(10, 6))

    # Sort and take top 10
    top_10 = df.nlargest(10, 'score').sort_values('score', ascending=True)

    # Shorten titles longer than 50 characters
    display_titles = [t[:47] + "..." if len(t) > 50 else t for t in top_10['title']]

    plt.barh(display_titles, top_10['score'], color='skyblue')
    plt.title('Top 10 Stories by Score')
    plt.xlabel('Score')
    plt.ylabel('Story Title')
    plt.tight_layout()

    plt.savefig(f"{output_dir}/chart1_top_stories.png")
    plt.close() # Close to free memory and avoid overlapping in dashboard
    print("Saved Chart 1: Top 10 Stories")

    # --- Task 3: Chart 2 - Stories per Category (6 marks) ---
    plt.figure(figsize=(8, 6))

    cat_counts = df['category'].value_counts()
    # Using a list of distinct colors
    colors = ['gold', 'lightgreen', 'coral', 'lightskyblue', 'orchid']

    cat_counts.plot(kind='bar', color=colors[:len(cat_counts)])
    plt.title('Distribution of Stories per Category')
    plt.xlabel('Category')
    plt.ylabel('Number of Stories')
    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.savefig(f"{output_dir}/chart2_categories.png")
    plt.close()
    print("Saved Chart 2: Stories per Category")

    # --- Task 4: Chart 3 - Score vs Comments (6 marks) ---
    plt.figure(figsize=(8, 6))

    # Split data for legend coloring
    popular = df[df['is_popular'] == True]
    not_popular = df[df['is_popular'] == False]

    plt.scatter(not_popular['score'], not_popular['num_comments'],
                alpha=0.6, label='Not Popular', c='gray')
    plt.scatter(popular['score'], popular['num_comments'],
                alpha=0.8, label='Popular', c='crimson', edgecolors='black')

    plt.title('Score vs. Number of Comments')
    plt.xlabel('Score')
    plt.ylabel('Comments')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    plt.savefig(f"{output_dir}/chart3_scatter.png")
    plt.close()
    print("Saved Chart 3: Score vs Comments")

    # --- Bonus: Dashboard Figure (Combining them) ---
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('TrendPulse Data Analysis Dashboard', fontsize=20)

    # Subplot 1: Top Stories
    axes[0, 0].barh(display_titles, top_10['score'], color='skyblue')
    axes[0, 0].set_title('Top 10 Stories')

    # Subplot 2: Category Bar
    cat_counts.plot(kind='bar', color=colors[:len(cat_counts)], ax=axes[0, 1])
    axes[0, 1].set_title('Stories per Category')

    # Subplot 3: Scatter Plot
    axes[1, 0].scatter(not_popular['score'], not_popular['num_comments'], alpha=0.5, c='gray')
    axes[1, 0].scatter(popular['score'], popular['num_comments'], alpha=0.7, c='crimson')
    axes[1, 0].set_title('Score vs Comments')
    axes[1, 0].set_xlabel('Score')
    axes[1, 0].set_ylabel('Comments')

    # Subplot 4: Summary Table (Empty or basic text)
    axes[1, 1].axis('off')
    summary_text = (
        f"Total Stories: {len(df)}\n"
        f"Avg Score: {df['score'].mean():.1f}\n"
        f"Most Active: {cat_counts.index[0]}\n"
        f"Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d')}"
    )
    axes[1, 1].text(0.5, 0.5, summary_text, fontsize=14, ha='center', va='center',
                    bbox=dict(facecolor='white', alpha=0.5))

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f"{output_dir}/dashboard.png")
    plt.close()

    print("--- Task 4 Complete ---")
    print(f" All 3 charts and the dashboard saved to the '{output_dir}/' folder.")

if __name__ == "__main__":
    run_visualization()

Created outputs/ folder.
Saved Chart 1: Top 10 Stories
Saved Chart 2: Stories per Category
Saved Chart 3: Score vs Comments
--- Task 4 Complete ---
 All 3 charts and the dashboard saved to the 'outputs/' folder.


Load data/trends_analysed.csv into a DataFrame
Create a folder called outputs/ if it doesn't exist
Use plt.savefig() before any plt.show() on all charts
#Chart 1: Top 10 Stories by Score
#Chart 2: Stories per Category
#Chart 3: Score vs Comments


In [13]:
#bonus
import pandas as pd
import matplotlib.pyplot as plt
import os

def run_visualization():
    # --- Task 1: Setup ---
    input_file = "data/trends_analysed.csv"
    if not os.path.exists(input_file):
        print("Error: trends_analysed.csv not found. Run Task 3 first.")
        return

    df = pd.read_csv(input_file)

    if not os.path.exists('outputs'):
        os.makedirs('outputs')

    # Prepare Data for Charts
    top_10 = df.nlargest(10, 'score').sort_values('score')
    display_titles = [t[:47] + "..." if len(t) > 50 else t for t in top_10['title']]
    cat_counts = df['category'].value_counts()
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#f1c40f', '#9b59b6']

    # --- Individual Charts (Tasks 2, 3, & 4) ---

    # Chart 1: Top 10 Stories
    plt.figure(figsize=(10, 6))
    plt.barh(display_titles, top_10['score'], color='#3498db')
    plt.title('Top 10 Stories by Score')
    plt.tight_layout()
    plt.savefig('outputs/chart1_top_stories.png')
    plt.close()

    # Chart 2: Categories
    plt.figure(figsize=(8, 6))
    cat_counts.plot(kind='bar', color=colors)
    plt.title('Stories per Category')
    plt.tight_layout()
    plt.savefig('outputs/chart2_categories.png')
    plt.close()

    # Chart 3: Scatter Plot
    plt.figure(figsize=(8, 6))
    for label, color in zip([True, False], ['#e74c3c', '#95a5a6']):
        subset = df[df['is_popular'] == label]
        plt.scatter(subset['score'], subset['num_comments'], label=f'Popular: {label}', c=color, alpha=0.7)
    plt.title('Score vs Comments')
    plt.legend()
    plt.tight_layout()
    plt.savefig('outputs/chart3_scatter.png')
    plt.close()

    # --- Dashboard: Combine All (Bonus +3 Marks) ---
    # Using a 2x2 grid for a balanced look
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('TrendPulse Dashboard', fontsize=24, fontweight='bold', y=0.98)

    # Position 1 (Top Left): Horizontal Bar
    axes[0, 0].barh(display_titles, top_10['score'], color='#3498db')
    axes[0, 0].set_title('Top 10 Stories by Score')
    axes[0, 0].set_xlabel('Score')

    # Position 2 (Top Right): Category Bar
    cat_counts.plot(kind='bar', color=colors, ax=axes[0, 1])
    axes[0, 1].set_title('Stories per Category')
    axes[0, 1].set_ylabel('Count')

    # Position 3 (Bottom Left): Scatter Plot
    for label, color in zip([True, False], ['#e74c3c', '#95a5a6']):
        subset = df[df['is_popular'] == label]
        axes[1, 0].scatter(subset['score'], subset['num_comments'], label=f'Popular: {label}', c=color, alpha=0.6)
    axes[1, 0].set_title('Engagement: Score vs Comments')
    axes[1, 0].set_xlabel('Score')
    axes[1, 0].set_ylabel('Comments')
    axes[1, 0].legend()

    # Position 4 (Bottom Right): Text Summary
    axes[1, 1].axis('off') # Hide axes for text display
    summary_info = (
        f"DATA SUMMARY\n"
        f"-----------------\n"
        f"Total Stories: {len(df)}\n"
        f"Highest Score: {df['score'].max()}\n"
        f"Avg Engagement: {df['engagement'].mean():.2f}\n"
        f"Most Active: {cat_counts.index[0].title()}"
    )
    axes[1, 1].text(0.5, 0.5, summary_info, fontsize=16, ha='center', va='center',
                    bbox=dict(facecolor='none', edgecolor='#34495e', boxstyle='round,pad=1'))

    # Final Layout Adjustment
    plt.subplots_adjust(top=0.90, hspace=0.3, wspace=0.3)

    # Save the final dashboard
    plt.savefig('outputs/dashboard.png', dpi=300)
    plt.close()

    print("--- Task 4 Complete ---")
    print(" Created: chart1_top_stories.png, chart2_categories.png, chart3_scatter.png")
    print(" Created Bonus: dashboard.png")

if __name__ == "__main__":
    run_visualization()
    #Combine all 3 charts into one figure:

--- Task 4 Complete ---
 Created: chart1_top_stories.png, chart2_categories.png, chart3_scatter.png
 Created Bonus: dashboard.png


--- Task 4 Complete ---
 Created: chart1_top_stories.png, chart2_categories.png, chart3_scatter.png
 Created Bonus: dashboard.png.
 #├── data/
│   ├── trends_20260404.json
│   ├── trends_clean.csv
│   └── trends_analysed.csv
├── outputs/
│   ├── chart1_top_stories.png
│   ├── chart2_categories.png
│   ├── chart3_scatter.png
│   └── dashboard.png
├── task1_data_collection.py
├── task2_data_processing.py
├── task3_analysis.py
└── task4_visualization.py

outputs/
├── chart1_top_stories.png
├── chart2_categories.png
├── chart3_scatter.png
└── dashboard.png  (bonus)


### Steps to Run the Pipeline
1. Run the **Task 1** cell to collect JSON data.
2. Verify the file exists in the `/data` folder.
3. Run **Task 2** to clean the data and output a CSV.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

steps = [
    "Step 1: Task 1 generates the JSON file.",
    "Step 2: Task 2 cleans the JSON and makes a CSV.",
    "Step 3: Task 3 calculates the statistics.",
    "Step 4: Task 4 creates the final Dashboard!"
]

button = widgets.Button(description="Show Next Step")
output = widgets.Output()
counter = 0

def on_button_clicked(b):
    global counter
    with output:
        if counter < len(steps):
            print(steps[counter])
            counter += 1
        else:
            print(" All steps completed!")

button.on_click(on_button_clicked)
display(button, output)

Button(description='Show Next Step', style=ButtonStyle())

Output()

#The pipeline mini project successfully executed and completed